In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OrdinalEncoder,
    OneHotEncoder,
    StandardScaler
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [ ]:
train_df = pd.read_csv("customer_churn_dataset-training-master.csv")
test_df = pd.read_csv("customer_churn_dataset-testing-master.csv")

In [ ]:
# -----------------------------
# 3. REMOVE USELESS COLUMN
# -----------------------------

train_df = train_df.drop('CustomerID', axis=1)
test_df = test_df.drop('CustomerID', axis=1)

In [ ]:
# 4. HANDLE MISSING VALUES
# -----------------------------

# Fill categorical columns using mode

for col in train_df.select_dtypes(include='object').columns:
    train_df[col] = train_df[col].fillna(
        train_df[col].mode()[0]
    )

for col in test_df.select_dtypes(include='object').columns:
    test_df[col] = test_df[col].fillna(
        test_df[col].mode()[0]
    )

# Fill numerical columns using median

for col in train_df.select_dtypes(include=['int64', 'float64']).columns:
    train_df[col] = train_df[col].fillna(
        train_df[col].median()
    )

for col in test_df.select_dtypes(include=['int64', 'float64']).columns:
    test_df[col] = test_df[col].fillna(
        test_df[col].median()
    )


In [ ]:
# -----------------------------
# 5. CHECK NULL VALUES
# -----------------------------

print("Remaining Null Values in Train:")
print(train_df.isnull().sum())

print("\nRemaining Null Values in Test:")
print(test_df.isnull().sum())

In [ ]:
# -----------------------------
# 6. DEFINE FEATURES & TARGET
# -----------------------------

X_train = train_df.drop('Churn', axis=1)
y_train = train_df['Churn']

X_test = test_df.drop('Churn', axis=1)
y_test = test_df['Churn']

In [ ]:
# -----------------------------
# 7. CATEGORY ORDER
# -----------------------------

subscription_order = ['Basic', 'Standard', 'Premium']

contract_order = ['Monthly', 'Quarterly', 'Annual']

In [ ]:
# -----------------------------
# 8. NUMERICAL FEATURES
# -----------------------------

numerical_features = [
    'Age',
    'Tenure',
    'Usage Frequency',
    'Support Calls',
    'Payment Delay',
    'Total Spend',
    'Last Interaction'
]

In [ ]:
# -----------------------------
# 9. PREPROCESSOR
# -----------------------------

preprocessor = ColumnTransformer(
    transformers=[

        # Scale Numerical Features
        (
            'num',
            StandardScaler(),
            numerical_features
        ),

        # Ordinal Encoding
        (
            'ordinal_sub',
            OrdinalEncoder(categories=[subscription_order]),
            ['Subscription Type']
        ),
        (
            'ordinal_con',
            OrdinalEncoder(categories=[contract_order]),
            ['Contract Length']
        ),

        # One Hot Encoding
        (
            'onehot_gender',
            OneHotEncoder(
                sparse_output=False,
                handle_unknown='ignore'
            ),
            ['Gender']
        )

    ],

    remainder='passthrough'
)

In [ ]:
# -----------------------------
# 10. APPLY PREPROCESSING
# -----------------------------

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [ ]:
# -----------------------------
# 11. CHECK FOR NaN VALUES
# -----------------------------

print("\nNaN values in X_train_processed:")
print(np.isnan(X_train_processed).sum())

print("\nNaN values in X_test_processed:")
print(np.isnan(X_test_processed).sum())


In [ ]:
# -----------------------------
# 12. TRAIN MODEL
# -----------------------------

model = LogisticRegression(max_iter=1000)

model.fit(X_train_processed, y_train)

In [ ]:
# -----------------------------
# 13. MAKE PREDICTIONS
# -----------------------------

y_pred = model.predict(X_test_processed)

In [ ]:
# -----------------------------
# 14. ACCURACY SCORE
# -----------------------------

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy Score:")
print(accuracy)

In [ ]:
# -----------------------------
# 15. CONFUSION MATRIX
# -----------------------------

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

In [ ]:
# -----------------------------
# 16. CLASSIFICATION REPORT
# -----------------------------

report = classification_report(y_test, y_pred)

print("\nClassification Report:")
print(report)

In [ ]:
# -----------------------------
# 17. FEATURE COEFFICIENTS
# -----------------------------

feature_names = preprocessor.get_feature_names_out()

coefficients = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model.coef_[0]
})

coefficients = coefficients.sort_values(
    by='Coefficient',
    ascending=False
)

print("\nFeature Coefficients:")
print(coefficients)

In [ ]:
# -----------------------------
# 18. TRAIN VS TEST SCORE
# -----------------------------

train_score = model.score(X_train_processed, y_train)

test_score = model.score(X_test_processed, y_test)

print("\nTrain Score:", train_score)

print("Test Score:", test_score)